# Ноутбук для проверки качества данных.

In [1]:
import pandas as pd

merchants = pd.read_csv("../data/merchants.csv", parse_dates=["created_at"])
sales_reps = pd.read_csv("../data/sales_reps.csv")
events = pd.read_csv("../data/stage_events.csv", parse_dates=["stage_ts"])
events.head()

,merchant_id,stage,stage_ts
0,M00001,lead,2026-01-01 00:05:09
1,M00001,qualified,2026-01-01 05:59:05
2,M00001,application,2026-01-03 00:01:58
3,M00001,kyc_passed,2026-01-04 20:31:07
4,M00001,contract_signed,2026-01-07 04:05:10


### 1. Смотрим на таблицу и ее колонки

In [2]:
merchants.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   merchant_id             30000 non-null  str           
 1   created_at              30000 non-null  datetime64[us]
 2   channel                 30000 non-null  str           
 3   size                    30000 non-null  str           
 4   cms                     30000 non-null  str           
 5   region                  29576 non-null  str           
 6   is_sales_led            30000 non-null  bool          
 7   rep_id                  7959 non-null   str           
 8   is_slow                 30000 non-null  bool          
 9   first_contact_hours     7959 non-null   float64       
 10  site_has_offer          30000 non-null  bool          
 11  site_has_refund_policy  30000 non-null  bool          
 12  kyc_reject_reason       6534 non-null   str           
 1

### 2. Проверка на пропуски

In [3]:
merchants.isna().sum()

merchant_id                   0
created_at                    0
channel                       0
size                          0
cms                           0
region                      424
is_sales_led                  0
rep_id                    22041
is_slow                       0
first_contact_hours       22041
site_has_offer                0
site_has_refund_policy        0
kyc_reject_reason         23466
last_stage                    0
activated                     0
turnover_30d_rub          24293
dtype: int64

In [4]:
print(merchants["merchant_id"].duplicated().sum())
print(events.duplicated(["merchant_id", "stage"]).sum())

0
0


Вывод по пропускам и дублям:
- пропуски в rep_id, first_contact_hours, kyc_reject_reason, turnover_30d_rub ожидаемы: они пустые у тех, кого это не касается (без сейлза, не отклонен, не активирован);
- пропуски в region (424 шт, около 1,4%) случайные, оставляю как есть
- Дубликатов в таблице merchants и events не обнаружено

### 3. Смотрим диапазон дат

In [5]:
print("Начало диапазона дат в таблице merchant:", merchants["created_at"].min(), "Конец диапазона дат в таблице merchant:", merchants["created_at"].max())
print("Начало диапазона дат в таблице events:", events["stage_ts"].min(), "Конец диапазона дат в таблице events:", events["stage_ts"].max())

Начало диапазона дат в таблице merchant: 2026-01-01 00:05:09 Конец диапазона дат в таблице merchant: 2026-09-29 23:53:08
Начало диапазона дат в таблице events: 2026-01-01 00:05:09 Конец диапазона дат в таблице events: 2026-10-29 18:00:37


Мерчанты созданы до 29 сентября, а события идут до 29 октября. Можно сделать вывод: мерчанты последних недель могли еще не успеть дойти до первого платежа, поэтому их конверсия будет занижена.

### 4. Значения в категориях

In [6]:
for col in ["channel", "size", "cms", "region"]:
    print(merchants[col].value_counts(dropna=False), "\n")

channel
organic_site      10126
partner            7203
outbound_sales     6066
referral           3568
ads                3037
Name: count, dtype: int64 

size
ip_micro      17404
smb           10169
enterprise     2427
Name: count, dtype: int64 

cms
custom_api      7367
tilda           7155
payment_link    6429
bitrix          5114
insales         3935
Name: count, dtype: int64 

region
moscow_spb          11299
central              6489
volga                4712
south                3554
siberia_far_east     3522
NaN                   424
Name: count, dtype: int64 



### 5. Проверка есть ли у каждого merchants свое событие lead в events

In [7]:
leads = events.loc[events["stage"] == "lead", "merchant_id"]
print(merchants["merchant_id"].isin(leads).all())

True


In [8]:
print(events["merchant_id"].isin(merchants["merchant_id"]).all())

True


Обе проверки дали True: таблицы согласованы, у каждого мерчанта есть старт воронки, лишних событий нет, можно считать конверсии от lead

### 6. Проверка логики этапов

In [9]:
order = ["lead", "qualified", "application", "kyc_passed", "contract_signed", "integration_done", "test_payment", "first_payment"]
events["stage_order"] = events["stage"].map({s: i for i, s in enumerate(order)})
events = events.sort_values(["merchant_id", "stage_order"])

In [10]:
last = events.groupby("merchant_id")["stage"].last()
print(merchants.set_index("merchant_id")["last_stage"].eq(last).all())

True


In [11]:
diff = events.groupby("merchant_id")["stage_ts"].diff()
print((diff < pd.Timedelta(0)).sum())

0


Данные согласованы и им можно доверять.